In [2]:
import cv2
from pathlib import Path
import pandas as pd
import numpy as np
from skimage.draw import polygon
import matplotlib.pyplot as plt
import torch
import lightning as L
from lightning.pytorch.loggers import TensorBoardLogger
import sys
# sys.path.insert(0, '../../echonet/src')
sys.path.insert(0, '../../src')
import utils
import datasets.echonet
import vids_segm_cld.vids_segm_pl as vids
import random
import argparse

from lightning.pytorch import seed_everything

In [ ]:
# Define paths 
path_to_data_train = '/project/home/pfischer95/Documents/echonet_dynamic_preprocessed/TRAIN/'
path_to_data_val = '/project/home/pfischer95/Documents/echonet_dynamic_preprocessed/VAL/'
path_to_data_test = '/project/home/pfischer95/Documents/echonet_dynamic_preprocessed/TEST/'

In [ ]:
# parameters
kl_weight = 0.005
var_penalty = 0.0
n_envs = 10
env_size_train = 1000
env_size_test = 20

In [ ]:
# set seed
seed_everything(42, workers=True)

In [ ]:
# loaders
loader_train = datasets.echonet.load_data_into_loader(64, path_to_data_train, shuffle=True)
loader_val = datasets.echonet.load_data_into_loader(64, path_to_data_val, shuffle=False)

In [ ]:
input_channels = 1 # one for bw images
num_classes = 2 # binary 
embedding_dim = 32

In [ ]:
print("Step 1: Pre-training embedding network...")
embedding_net = vids.UNetDenseEmbedding(
    n_channels=input_channels, 
    embedding_dim=embedding_dim, 
    bilinear=False
)

embedding_net, pretrained_model = vids.pretrain_segmentation_embedding(
    embedding_net=embedding_net,
    loader=loader_train,
    num_classes=num_classes,
    epochs=10,
    lr=1e-3
)

In [ ]:
# For segmentation with small theta, we need small inference network
theta_dim = embedding_dim * num_classes + num_classes  # 32*2 + 2 = 66
inference_hidden = [256, 128, 64]  # Smaller than default

vids_model = vids.VIDS(
    embedding_net=embedding_net,
    embedding_dim=embedding_dim,
    output_dim=num_classes,
    task="segmentation",
    inference_hidden_dims=inference_hidden,
    kl_weight=kl_weight,
    variance_penalty=var_penalty,
    num_classes=num_classes,
    learning_rate=1e-4,
    num_environments=n_envs,
    env_train_size=env_size_train,
    env_test_size=env_size_test,
    num_prediction_samples=20,
)
print(f"  Prediction head params (θ): {vids_model.prediction_head.num_params}")
print(f"  Inference network params: {sum(p.numel() for p in vids_model.inference_net.parameters())}")

In [ ]:
print("\nStep 3: Training VIDS with Lightning Trainer...")
logger = TensorBoardLogger(
    save_dir="tb_logs", 
    name="adults"
)

In [ ]:
trainer = L.Trainer(
    max_epochs=50, 
    logger=logger,
    accelerator="gpu",
    devices=1
)


In [ ]:
# 7. Train
trainer.fit(model=vids_model, train_dataloaders=loader_train, val_dataloaders=loader_val)